In [18]:
import torch
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torch import nn
from pytorch_lightning.core import LightningModule
import pytorch_lightning as pyl
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from pickle import load
from rdkit.Chem import MolFromSmiles
from pandas import DataFrame, concat
from rdkit.Chem import AllChem 
from rdkit import DataStructs


In [19]:
class PotencyModel(LightningModule):
    def __init__(self):
        super().__init__()
        self.test_predictions = []
        self.targets = []
        
        self.fingerprint_fc = nn.Sequential(
            nn.Linear(2048, 1012),
            nn.LeakyReLU(),
            nn.Linear(1012, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 1)
        )

        self.loss_fn = nn.MSELoss()

    def forward(self, fingerprint):
        return self.fingerprint_fc(fingerprint)

    def training_step(self, batch, batch_idx):
        fingerprint, y = batch
        y_pred = self(fingerprint)

        loss = self.loss_fn(y_pred, y)
        self.log('Train MSE', loss, on_step=False, on_epoch=True, prog_bar=True)

        r2 = r2_score(y.detach().cpu().numpy(), y_pred.detach().cpu().numpy())
        self.log('Train R²', r2, on_step=False, on_epoch=True, prog_bar=True)

        return loss
    
    def validation_step(self, batch, batch_idx):
        fingerprint, y = batch
        y_pred = self(fingerprint)

        loss = self.loss_fn(y_pred, y)
        self.log('Validation MSE', loss, on_step=False, on_epoch=True, prog_bar=True)

        r2 = r2_score(y.cpu().numpy(), y_pred.cpu().numpy())
        self.log('Validation R²', r2, on_step=False, on_epoch=True, prog_bar=True)

        return loss
    
    def test_step(self, batch, batch_idx):
        fingerprint, y = batch
        y_pred = self(fingerprint)

        loss = self.loss_fn(y_pred, y)
        self.log('Test MSE', loss, on_step=False, on_epoch=True, prog_bar=True)
    
        self.test_predictions.extend(y_pred.cpu().numpy())
        self.targets.extend(y.cpu().numpy())
        
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

with open('../data/classifier/potency.pickle', 'rb') as file:
    potency = load(file)

In [20]:
def calc_morgan(mols):
    """ генерация молекулярных отпечатков по методу Моргана с радиусом 2 и длиной 2048
    """
    mfp_gen = AllChem.GetMorganGenerator(radius=2, ) 
    for_df = []
    for m in mols:
        arr = np.zeros((1,), dtype=int)
        DataStructs.ConvertToNumpyArray(mfp_gen.GetFingerprint(m), arr)
        for_df.append(arr)
    return DataFrame(for_df)

with open('../data/classifier/no_dubl_potency.pickle', 'rb') as file:
    potency = load(file)

In [21]:
molecules = [
    MolFromSmiles(mol) for mol in potency['SMILES']
]

In [ ]:
X = calc_morgan(molecules)
Y = potency['Potency']
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [27]:
train_loader = DataLoader(concat((x_train, y_train), axis=1).to_numpy(), batch_size=2)
test_loader = DataLoader(concat((x_test, y_test), axis=1).to_numpy(), batch_size=2)

In [28]:
model = PotencyModel()

trainer = pyl.Trainer(max_epochs=5, accelerator="auto")
trainer.fit(model, train_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/titovetsla/.conda/envs/prediction_of_LD50/lib/python3.13/site-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

  | Name           | Type       | Params | Mode 
------------------------------------------------------
0 | fingerprint_fc | Sequential | 2.6 M  | train
1 | loss_fn        | MSELoss    | 0      | train
------------------------------------------------------
2.6 M     Trainable params
0         Non-trainable params
2.6 M     Total params
10.371    Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode
/home/titovetsla/.

Epoch 0:   0%|          | 0/3302 [00:00<?, ?it/s] 

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float

In [ ]:
trainer.test(model, test_loader)

y_pred_test = model.test_predictions
y_true = model.targets
mse = mean_squared_error(y_true, y_pred_test)
q2 = r2_score(y_true, y_pred_test)
rmse = np.sqrt(mse)
print(f'test MSE: {mse:.4f}, test PRMSE: {rmse:.4f}, test Q²: {q2:.4f}')